## Inception, ResNet, and DenseNet

Goal:
- understand three major CNN architecture ideas
- build simplified versions from scratch
- compare their shapes and behaviors
- become comfortable reading architecture code

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


1. Why new architectures were needed

As CNNs got deeper, people ran into problems:
- training got harder
- gradients became harder to pass through many layers
- model design became more important

Different architectures tried different solutions:
- Inception: multiple filter sizes in parallel
- ResNet: skip connections
- DenseNet: connect each layer to later layers

In [2]:
# Fake image input for architecture experiments

x = torch.randn(4,3,32,32)
print(x.shape)

torch.Size([4, 3, 32, 32])


Inception idea

Instead of choosing just one filter size,
use multiple branches in parallel.

For example:
- 1x1 conv
- 3x3 conv
- 5x5 conv
- pooling branch

Then concatenate outputs along channels.


In [3]:
# Simple Inception block
class SimpleInceptionBlock(nn.Module):
    def __init__(self, in_channels, out1, out3, out5, out_pool):
        super().__init__()

        self.branch1 = nn.Conv2d(in_channels, out1, kernel_size=1)

        self.branch3 = nn.Conv2d(in_channels, out3, kernel_size=3, padding=1)

        self.branch5 = nn.Conv2d(in_channels, out5, kernel_size=5, padding=2)

        self.branch_pool = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_pool, kernel_size=1)
        )

    def forward(self, x):
        b1 = torch.relu(self.branch1(x))
        b3 = torch.relu(self.branch3(x))
        b5 = torch.relu(self.branch5(x))
        bp = torch.relu(self.branch_pool(x))

        out = torch.cat([b1, b3, b5, bp], dim=1)
        return out

In [5]:
inception_block = SimpleInceptionBlock(
    in_channels=3,
    out1=4,
    out3=4,
    out5=4,
    out_pool=4
)

y = inception_block(x)

print("input shape:", x.shape)
print("output shape:", y.shape)

input shape: torch.Size([4, 3, 32, 32])
output shape: torch.Size([4, 16, 32, 32])


Why channel count increased

We concatenated 4 branches.
Each branch outputs 4 channels.

Total output channels:
4 + 4 + 4 + 4 = 16

In [6]:
print(y.shape)

torch.Size([4, 16, 32, 32])


More efficient Inception idea

A 1x1 convolution can reduce channels before expensive convolutions.
This makes computation cheaper.

This was an important design idea in Inception networks.


In [8]:
class BetterInceptionBlock(nn.Module):
    def __init__(self, in_channels, reduce3, out3, reduce5, out5, out1, out_pool):
        super().__init__()

        self.branch1 = nn.Conv2d(in_channels, out1, kernel_size=1)

        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, reduce3, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(reduce3, out3, kernel_size=3, padding=1)
        )

        self.branch5 = nn.Sequential(
            nn.Conv2d(in_channels, reduce5, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(reduce5, out5, kernel_size=5, padding=2)
        )

        self.branch_pool = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_pool, kernel_size=1)
        )

    def forward(self, x):
        b1 = torch.relu(self.branch1(x))
        b3 = self.branch3(x)
        b5 = self.branch5(x)
        bp = torch.relu(self.branch_pool(x))

        out = torch.cat([b1, b3, b5, bp], dim=1)
        return out

In [9]:
better_inception = BetterInceptionBlock(
    in_channels=3,
    reduce3=4,
    out3=8,
    reduce5=2,
    out5=4,
    out1=4,
    out_pool=4
)

y = better_inception(x)

print("input shape:", x.shape)
print("output shape:", y.shape)

input shape: torch.Size([4, 3, 32, 32])
output shape: torch.Size([4, 20, 32, 32])


## Build a tiny Inception-style network

In [10]:
class TinyInceptionNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.inception1 = BetterInceptionBlock(
            in_channels=8,
            reduce3=4,
            out3=8,
            reduce5=2,
            out5=4,
            out1=4,
            out_pool=4
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(20, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.inception1(x)
        x = self.pool(x)
        x = x.view(x.shape[0], -1)
        x = self.fc(x)
        return x

In [11]:
model_inception = TinyInceptionNet(num_classes=10)
y = model_inception(x)

print("input shape:", x.shape)
print("output shape:", y.shape)
print(model_inception)

input shape: torch.Size([4, 3, 32, 32])
output shape: torch.Size([4, 10])
TinyInceptionNet(
  (stem): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
  )
  (inception1): BetterInceptionBlock(
    (branch1): Conv2d(8, 4, kernel_size=(1, 1), stride=(1, 1))
    (branch3): Sequential(
      (0): Conv2d(8, 4, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU()
      (2): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (branch5): Sequential(
      (0): Conv2d(8, 2, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU()
      (2): Conv2d(2, 4, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    )
    (branch_pool): Sequential(
      (0): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
      (1): Conv2d(8, 4, kernel_size=(1, 1), stride=(1, 1))
    )
  )
  (pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Linear(in_features=20, out_features=10, bias=True)
)


## ResNet idea

Instead of only learning:
F(x)

learn:
F(x) + x

This is called a residual connection or skip connection.

The shortcut helps gradients and information flow better.

In [12]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = torch.relu(out)
        out = self.conv2(out)

        out = out + identity
        out = torch.relu(out)

        return out

In [13]:
x_res = torch.randn(4, 8, 32, 32)
res_block = ResidualBlock(channels=8)

y_res = res_block(x_res)

print("input shape:", x_res.shape)
print("output shape:", y_res.shape)

input shape: torch.Size([4, 8, 32, 32])
output shape: torch.Size([4, 8, 32, 32])


 Residual block with channel change

If input and output shapes differ,
the shortcut path must also be transformed.


In [14]:
class ResidualBlockProj(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = torch.relu(out)
        out = self.conv2(out)

        out = out + identity
        out = torch.relu(out)

        return out


In [15]:
x = torch.randn(4, 8, 32, 32)
proj_block = ResidualBlockProj(in_channels=8, out_channels=16, stride=2)

y = proj_block(x)

print("input shape:", x.shape)
print("output shape:", y.shape)

input shape: torch.Size([4, 8, 32, 32])
output shape: torch.Size([4, 16, 16, 16])


## DenseNet idea

Instead of adding previous features,
concatenate them.

Each layer receives all earlier feature maps.

This encourages feature reuse.

In [16]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()

        self.conv = nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1)

    def forward(self, x):
        new_features = torch.relu(self.conv(x))
        out = torch.cat([x, new_features], dim=1)
        return out

In [17]:
x = torch.randn(4, 8, 32, 32)
dense_layer = DenseLayer(in_channels=8, growth_rate=4)

y = dense_layer(x)

print("input shape:", x.shape)
print("output shape:", y.shape)

input shape: torch.Size([4, 8, 32, 32])
output shape: torch.Size([4, 12, 32, 32])


Why channels grow in DenseNet

If input has 8 channels
and growth rate is 4,
output has:

8 + 4 = 12 channels

## Dense block
Multiple dense layers stacked together.

In [18]:
class DenseBlock(nn.Module):
    def __init__(self, in_channels, growth_rate, num_layers):
        super().__init__()

        layers = []
        current_channels = in_channels

        for _ in range(num_layers):
            layers.append(DenseLayer(current_channels, growth_rate))
            current_channels = current_channels + growth_rate

        self.layers = nn.ModuleList(layers)
        self.out_channels = current_channels

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [20]:
dense_block = DenseBlock(in_channels=8, growth_rate=4, num_layers=3)

x = torch.randn(4, 8, 32, 32)
y = dense_block(x)

print("input shape:", x.shape)
print("output shape:", y.shape)
print("out channels:", dense_block.out_channels)

input shape: torch.Size([4, 8, 32, 32])
output shape: torch.Size([4, 20, 32, 32])
out channels: 20


## Transition layer

DenseNet often uses transition layers to reduce channels and spatial size.

In [21]:
class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )

    def forward(self, x):
        return self.layers(x)

In [22]:
transition = TransitionLayer(in_channels=20, out_channels=10)

x = torch.randn(4, 20, 32, 32)
y = transition(x)

print("input shape:", x.shape)
print("output shape:", y.shape)

input shape: torch.Size([4, 20, 32, 32])
output shape: torch.Size([4, 10, 16, 16])


## Compare parameter counts
Production habit:
compare model size, not just ideas.

In [24]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("TinyInceptionNet params:", count_parameters(model_inception))


TinyInceptionNet params: 1060
